## Ray Actor + 一致性 ADMM 实现分布式 Elastic Net

求解弹性网（Elastic Net）模型：

$$
\min_x\ \frac{1}{2}\Vert Ax-b\Vert_2^2 + \lambda \left(\alpha\Vert x\Vert_1+\frac{1}{2}(1-\alpha)\Vert x\Vert^2\right).
$$

它与 Lasso 的主要区别就是在正则项中增加了二次项 $\Vert x\Vert^2$。Lasso 是弹性网 $\alpha=1$ 的特殊情形。

定义 $g(z)=\alpha\Vert z\Vert_1+(1-\alpha)\Vert z\Vert^2/2$，已知 $g(z)$ 的近端算子为
$$
\operatorname{prox}_{\lambda g}(x) = \underset{z}{\arg\min} \left\{ g(z) + \frac{1}{2\lambda} \Vert z - x\Vert^2 \right\}=\frac{1}{1 + \lambda(1-\alpha)} \, S_{\lambda\alpha}(x).
$$

其中 $S_{\kappa}(x) = \mathrm{sign}(x)\cdot \max(|x| - \kappa, 0)$ 是 Soft-thresholding 算子。

In [1]:
# 请在此处完成代码
import os
import time

import numpy as np
import pandas as pd

import ray
from scipy.linalg import cho_factor, cho_solve

if ray.is_initialized():
    ray.shutdown()

ray.init(
    ignore_reinit_error=True,
    include_dashboard=False,
)

2026-06-20 12:42:31,141	INFO worker.py:2012 -- Started a local Ray instance.
/home/limengli/anaconda3/envs/ray/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.13
Ray version:,2.55.1


In [2]:
def make_sparse_regression(n=200000, p=100, s=10, noise=0.5, seed=123):
    rng = np.random.default_rng(seed)

    A = rng.normal(size=(n, p))
    A = (A - A.mean(axis=0)) / A.std(axis=0)

    beta_true = np.zeros(p)
    support = rng.choice(p, size=s, replace=False)
    beta_true[support] = rng.normal(loc=0.0, scale=3.0, size=s)

    b = A @ beta_true + noise * rng.normal(size=n)

    return A, b, beta_true


n_obs = 10000
n_features = 50
n_nonzero = 8

A, b, beta_true = make_sparse_regression(
    n=n_obs,
    p=n_features,
    s=n_nonzero,
    noise=0.5,
    seed=123
)

df = pd.DataFrame(A, columns=[f"x_{j}" for j in range(n_features)])
df.insert(0, "b", b)

file_path = "elastic_net_data.csv"
df.to_csv(file_path, index=False)

print("data shape:", df.shape)
print("file size MB:", round(os.path.getsize(file_path) / 1024 / 1024, 2))

data shape: (10000, 51)
file size MB: 9.54


In [3]:
def soft_threshold(x, kappa):
    return np.sign(x) * np.maximum(np.abs(x) - kappa, 0.0)


def elastic_net_prox(x, tau, alpha):
    return soft_threshold(x, tau * alpha) / (1.0 + tau * (1.0 - alpha))

In [4]:
@ray.remote(num_cpus=1)
class ElasticNetADMMWorker:
    def __init__(self, A_i, b_i, rho, worker_id=None):
        self.A_i = A_i
        self.b_i = b_i
        self.rho = rho
        self.worker_id = worker_id

        self.n_i, self.p = self.A_i.shape
        self.x_i = np.zeros(self.p)
        self.u_i = np.zeros(self.p)

        self.M = self.A_i.T @ self.A_i + self.rho * np.eye(self.p)
        self.q = self.A_i.T @ self.b_i
        
        self.M_factor = cho_factor(self.M, lower=True, check_finite=False)#M在迭代中不变，先分解一次，后面每轮直接复用

    def x_update(self, z):
        rhs = self.q + self.rho * (z - self.u_i)
        self.x_i = cho_solve(self.M_factor, rhs, check_finite=False)
        return self.x_i.copy(), self.u_i.copy()

    def u_update(self, z):
        self.u_i = self.u_i + self.x_i - z
        return float(np.linalg.norm(self.x_i - z) ** 2)

    def loss_at(self, z):
        r = self.A_i @ z - self.b_i
        return float(0.5 * np.dot(r, r))

    def get_state(self):
        return {
            "worker_id": self.worker_id,
            "n_i": self.n_i,
            "p": self.p,
            "x_i": self.x_i.copy(),
            "u_i": self.u_i.copy(),
        }

In [5]:
chunk_size = 1000
rho = 10.0

workers = []
n_rows_total = 0

for block_id, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size)):
    b_chunk = chunk["b"].to_numpy(dtype=np.float64)
    A_chunk = chunk.drop(columns=["b"]).to_numpy(dtype=np.float64)

    n_rows_total += len(chunk)

    A_ref = ray.put(A_chunk)
    b_ref = ray.put(b_chunk)

    worker = ElasticNetADMMWorker.remote(A_ref, b_ref, rho, worker_id=block_id)
    workers.append(worker)

print("number of workers:", len(workers))
print("total rows:", n_rows_total)
print("p:", n_features)

number of workers: 10
total rows: 10000
p: 50


In [ ]:
"""states = ray.get([w.get_state.remote() for w in workers])

for s in states[:5]:
    print(f"worker {s['worker_id']}: n_i={s['n_i']}, p={s['p']}")

print("...")"""

In [6]:
def distributed_elastic_net_admm(workers, p, lam, alpha, rho, max_iter=40, tol=1e-4):
    N = len(workers)
    z = np.zeros(p)
    history = []

    for k in range(max_iter):
        z_old = z.copy()

        xu_pairs = ray.get([w.x_update.remote(z) for w in workers])

        xs = np.stack([pair[0] for pair in xu_pairs], axis=0)
        us = np.stack([pair[1] for pair in xu_pairs], axis=0)

        x_bar = xs.mean(axis=0)
        u_bar = us.mean(axis=0)

        tau = lam / (N * rho)
        z = elastic_net_prox(x_bar + u_bar, tau, alpha)

        r2_list = ray.get([w.u_update.remote(z) for w in workers])
        r_norm = float(np.sqrt(np.sum(r2_list)))
        s_norm = float(rho * np.sqrt(N) * np.linalg.norm(z - z_old))

        if k % 10 == 0 or k == max_iter - 1:
            losses = ray.get([w.loss_at.remote(z) for w in workers])
            penalty = lam * (
                alpha * np.linalg.norm(z, ord=1)
                + 0.5 * (1.0 - alpha) * np.dot(z, z)
            )
            obj = float(np.sum(losses) + penalty)
        else:
            obj = np.nan

        history.append({
            "iter": k,
            "objective": obj,
            "r_norm": r_norm,
            "s_norm": s_norm,
            "nnz": int(np.sum(np.abs(z) > 1e-6)),
        })

        if k % 10 == 0 or k == max_iter - 1:
            print(
                f"iter={k:03d}, "
                f"obj={obj:.4f}, "
                f"r={r_norm:.4e}, "
                f"s={s_norm:.4e}, "
                f"nnz={np.sum(np.abs(z) > 1e-6)}"
            )

        if r_norm < tol and s_norm < tol:
            print("converged at iter", k)
            break

    return z, pd.DataFrame(history)

In [7]:
alpha = 0.5
lam = 0.001 * n_rows_total

start = time.time()

z_hat, hist = distributed_elastic_net_admm(
    workers=workers,
    p=n_features,
    lam=lam,
    alpha=alpha,
    rho=rho,
    max_iter=40,
    tol=1e-4
)

print("elapsed seconds:", round(time.time() - start, 2))
hist.tail()

iter=000, obj=2949.9077, r=1.5483e+00, s=2.1645e+02, nnz=8
iter=010, obj=1448.4031, r=3.2158e-01, s=1.7611e-01, nnz=26
iter=020, obj=1447.8424, r=2.8652e-01, s=2.5808e-02, nnz=35
iter=030, obj=1447.7177, r=2.5680e-01, s=1.8858e-03, nnz=39
iter=039, obj=1447.6877, r=2.3310e-01, s=1.6992e-03, nnz=40
elapsed seconds: 0.81


,iter,objective,r_norm,s_norm,nnz
35,35,NaN,0.243309,0.001778,40
36,36,NaN,0.240710,0.001760,40
37,37,NaN,0.238142,0.001739,40
38,38,NaN,0.235604,0.001719,40
39,39,1447.687676,0.233096,0.001699,40


In [8]:
true_support = np.sort(np.where(beta_true != 0)[0])
estimated_support = np.sort(np.where(np.abs(z_hat) > 1e-3)[0])

print("alpha:", alpha)
print("true support:     ", true_support)
print("estimated support:", estimated_support)

result = pd.DataFrame({
    "beta_true": beta_true,
    "beta_hat": z_hat,
    "abs_beta_hat": np.abs(z_hat),
})

result["selected"] = result["abs_beta_hat"] > 1e-3

result.sort_values("abs_beta_hat", ascending=False).head(15)

alpha: 0.5
true support:      [15 27 30 31 34 39 42 48]
estimated support: [ 0  2  3  4  6  7  8  9 10 11 12 13 14 15 16 17 18 19 23 24 25 27 28 30
 31 32 34 35 36 37 38 39 40 42 43 44 45 47 48 49]


,beta_true,beta_hat,abs_beta_hat,selected
27,-4.662044,-4.652892,4.652892,True
34,3.133512,3.130416,3.130416,True
42,2.630453,2.625958,2.625958,True
48,-2.450610,-2.453043,2.453043,True
31,2.165975,2.173971,2.173971,True
39,1.494072,1.493171,1.493171,True
15,1.352907,1.357730,1.357730,True
30,1.196274,1.187704,1.187704,True
43,0.000000,0.013485,0.013485,True
16,0.000000,0.011354,0.011354,True
